<a href="https://colab.research.google.com/github/Iddrisu-Abdulai/Analysis_Python/blob/main/SMS_Spam_Collection.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
#Loading Libraries
!pip install kaggle
!pip install fuzzywuzzy
!pip install Levenshtein
!pip install nltk
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score
import string
import os
from fuzzywuzzy import fuzz, process
from google.colab import drive
drive.mount('/content/drive')



Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


Saving kaggle.json to kaggle (2).json


{'kaggle (2).json': b'{"username":"iddrisuabdulai","key":"13eb2fa1d74b887098707400d621fb23"}'}

In [ ]:
# Move the kaggle.json file to the correct location
from google.colab import files
files.upload()
!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

# Download the dataset
!kaggle datasets download -d uciml/sms-spam-collection-dataset

# Unzip the downloaded file
!unzip sms-spam-collection-dataset.zip

#Load the data
df_train = pd.read_csv('spam.csv', encoding='latin-1')
df_test = pd.read_csv('spam.csv', encoding='latin-1')


# Print the column names to verify
print(df_train.columns)
print(df_test.columns)



Saving kaggle.json to kaggle (4).json
Dataset URL: https://www.kaggle.com/datasets/uciml/sms-spam-collection-dataset
License(s): unknown
sms-spam-collection-dataset.zip: Skipping, found more recently modified local copy (use --force to force download)
Archive:  sms-spam-collection-dataset.zip
replace spam.csv? [y]es, [n]o, [A]ll, [N]one, [r]ename: y
  inflating: spam.csv                
Index(['v1', 'v2', 'Unnamed: 2', 'Unnamed: 3', 'Unnamed: 4'], dtype='object')
Index(['v1', 'v2', 'Unnamed: 2', 'Unnamed: 3', 'Unnamed: 4'], dtype='object')


In [ ]:
# Rename the 'v2' column to 'message'
df_train = df_train.rename(columns={'v2': 'message'})
df_test = df_test.rename(columns={'v2': 'message'})

# Preprocess the data
def preprocess_text(text):
    text = text.lower()
    text = ''.join([char for char in text if char not in string.punctuation])
    words = text.split()
    return ' '.join(words)

df_train['message'] = df_train['message'].apply(preprocess_text)
df_test['message'] = df_test['message'].apply(preprocess_text)



In [ ]:
#Convert text data into numerical data: using TF-IDF Vectorization
vectorizer = TfidfVectorizer()
X_train = vectorizer.fit_transform(df_train['message'])
X_test = vectorizer.transform(df_test['message'])
y_train = df_train['v1']
y_test = df_test['v1']

In [ ]:
# Initialize and train the MultinomialNB classifier: using hte training data
model = MultinomialNB()

model.fit(X_train, y_train)

MultinomialNB()

In [ ]:
# Evaluate the model
y_pred = model.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)
print(f'Accuracy: {accuracy}')

Accuracy: 0.9716439339554918


In [ ]:
#Create the 'predict_message' function
def predict_message(message):
  message = preprocess_text(message)
  message_vector = vectorizer.transform([message])
  prediction = model.predict(message_vector)[0]
  prediction_proba = model.predict_proba(message_vector)[0]
  return [prediction_proba[1], 'spam' if prediction == 'spam' else 'ham']

In [ ]:
# Example usage
message = "Congratulations! You've won a $1000 gift card. Click here to claim now."
prediction = predict_message(message)
print(prediction)

[0.8370232402576359, 'spam']
